# Feature Serving

> **NB:** eseguire questa procedura solo quando è necessario aggiornare le feature utilizzate dal modello.

## Prerequisiti

- La tabella sorgente deve essere una **Delta Table** in Unity Catalog con una **Primary Key**.
- La tabella sorgente deve avere **Delta Change Data Feed (CDF)** abilitato (`delta.enableChangeDataFeed = true`).
## Procedura

1. **Pubblicare la Feature Table basata sulla tabella sorgente nell'Online Feature Store**
   - Materializzare la Feature Table nell'Online Feature Store affinché sia disponibile per il serving.

2. **Schedulare la sincronizzazione dell'Online Feature Table**
   - Creare un **Databricks Job (Lakeflow Job)** che esegua periodicamente la pubblicazione/sincronizzazione (`publish_table`) dall'offline Feature Table all'Online Feature Store.
   - Configurare la frequenza di aggiornamento in base alla latenza richiesta dal caso d'uso.
3. **Creare il FeatureSpec**
   - Definire le feature che il Feature Serving Endpoint dovrà recuperare.
   - Eventualmente aggiungere **Feature Functions** per calcolare feature on-demand durante l'inferenza.

4. **Creare il Feature Serving Endpoint**
   - Creare il Feature Serving Endpoint dalla UI (oppure tramite SDK o REST API).
   - Associare il `FeatureSpec` creato nel passo precedente.

5. **Verificare il funzionamento**
   - Testare l'endpoint passando le chiavi primarie richieste e verificando che vengano restituite le feature attese.

_____________________________________________________________________________________________________________________________________________________________________________________

**Documentazione Databricks**: https://learn.microsoft.com/en-us/azure/databricks/machine-learning/feature-store/feature-serving-tutorial

In [0]:
%run ./00_utility

In [0]:
%pip install databricks-feature-engineering==0.16.0

In [0]:
import re
import time
import mlflow.deployments
import pandas as pd

from databricks.feature_engineering import FeatureEngineeringClient, FeatureFunction,FeatureLookup
from databricks.feature_store import FeatureStoreClient

In [0]:
# Set-up catalog
catalog = get_catalog()
print("Catalog: ", catalog)
# Set-up catalog feature serving 
feature_serving_catalog = get_catalog_feature_serving()
print("Feature Serving catalog: ", feature_serving_catalog)

# Initialize the feature engineering client
fe = FeatureEngineeringClient()

# Initialize the feature store client
fs = FeatureStoreClient()

## Set-up feature table basata su "{catalog}.whatif.out_palinsesto_predict_all_slots"

### Pre-requisiti

In [0]:
# colonna della PK deve essere non nullable
spark.sql(f"""
ALTER TABLE `{catalog}`.`whatif`.`out_palinsesto_predict_all_slots`
ALTER COLUMN ID SET NOT NULL
""")

In [0]:
# impostare PK
spark.sql(f"""
ALTER TABLE `{catalog}`.`whatif`.`out_palinsesto_predict_all_slots`
ADD CONSTRAINT pk_pred_all_slots
PRIMARY KEY (ID)""")

In [0]:
# attivare ChangeDataFeed
spark.sql(f"""ALTER TABLE `{catalog}`.`whatif`.`out_palinsesto_predict_all_slots` SET TBLPROPERTIES (delta.enableChangeDataFeed = 'true')""")

### 1. Materializzazione Feature Table nell'Online Store

In [0]:
# check che il feature online store sia disponibile, se no creiamolo
store = fe.get_online_store(name=f"whatif-features-online-store{feature_serving_catalog}")
if store:
    print(f"Store: {store.name}, State: {store.state}")
else:
    store = fe.create_online_store(
    name=f"whatif-features-online-store{feature_serving_catalog}", # maximum of 63 bytes
    capacity="CU_2"  # Valid options: "CU_1", "CU_2", "CU_4", "CU_8"
)

In [0]:
# materializziamo la feature table nell'online store
fe.publish_table(
    online_store=store,
    source_table_name=f"{catalog}.whatif.out_palinsesto_predict_all_slots",
    online_table_name=f"{catalog}.whatif.out_palinsesto_predict_all_slots_feat"
)

### 2. Schedulazione della sincronizzazione della Feature Table con la tabella sorgente

Aggiunta la sincronizzazione come task al job WhatIf

### 3. Creazione del FeatureSpec

In [0]:
# definiamo il lookup per estrarre tutte le features del record (univoco) identificato dalla look up key (ID)
feature_lookups = [
    FeatureLookup(
        table_name=f"{catalog}.whatif.out_palinsesto_predict_all_slots",
        feature_names=None,  # Include all feature columns except lookup keys
        lookup_key=["ID"],
    ),
]

# creiamo il feature spec con la look up dentro
fe.create_feature_spec(
    name=f"{catalog}.whatif.out_palinsesto_predict_all_slots_features",
    features=feature_lookups
)

In [0]:
# testiamo che la registrazione del feature spec sia andato a buon fine
input_test_1 = pd.DataFrame(data = {'ID' : ['Rai 1_2026-07-05_rai news_06:00']})
print(f"Test ID: {input_test_1}")

feature_set_test_1 = fe.create_training_set(
    df=spark.createDataFrame(input_test_1),                                 
    feature_spec=f"{catalog}.whatif.out_palinsesto_predict_all_slots_features",
    label=None,
)

feature_set_test_1_df = feature_set_test_1.load_df()
print("Features:")
display(feature_set_test_1_df)

### 4. Creazione Feature Serving Endpoint

Su **Unity Catalog > catalog "{catalog}" > schema "what-id" > tab "Functions"**: selezionare la Feature Spec e poi cliccare in alto a destea su "Serve this Feature Spec"

### 5. Test Feature Serving Endpoint

In [0]:
client = mlflow.deployments.get_deploy_client("databricks")
response_test_1 = client.predict(endpoint="features_all_slots_preds", inputs={"dataframe_records": input_test_1.to_dict(orient="records")})
response_test_1_df = pd.DataFrame(response_test_1["outputs"])
display(response_test_1_df)

## Set-up feature table basata su "{catalog}.whatif.output_palinsesto_passato_enriched_feat"

### Pre-requisiti

In [0]:
# colonna della PK deve essere non nullable
spark.sql(f"""
ALTER TABLE `{catalog}`.`whatif`.`output_palinsesto_passato_enriched`
ALTER COLUMN Programma SET NOT NULL
""")

In [0]:
# impostare PK
spark.sql(f"""
ALTER TABLE `{catalog}`.`whatif`.`output_palinsesto_passato_enriched`
ADD CONSTRAINT pk_pred_all_passato_enriched
PRIMARY KEY (Programma)""")

In [0]:
# attivare ChangeDataFeed
spark.sql(f"""ALTER TABLE `{catalog}`.`whatif`.`output_palinsesto_passato_enriched` SET TBLPROPERTIES (delta.enableChangeDataFeed = 'true')""")

### 1. Materializzazione Feature Table nell'Online Store

In [0]:
# check che il feature online store sia disponibile, se no creiamolo
store = fe.get_online_store(name=f"whatif-features-online-store{feature_serving_catalog}")
if store:
    print(f"Store: {store.name}, State: {store.state}")
else:
    store = fe.create_online_store(
    name=f"whatif-features-online-store{feature_serving_catalog}", # maximum of 63 bytes
    capacity="CU_2"  # Valid options: "CU_1", "CU_2", "CU_4", "CU_8"
)

In [0]:
# materializziamo la feature table nell'online store
fe.publish_table(
    online_store=store,
    source_table_name=f"{catalog}.whatif.output_palinsesto_passato_enriched",
    online_table_name=f"{catalog}.whatif.output_palinsesto_passato_enriched_feat"
)

### 2. Schedulazione della sincronizzazione della Feature Table con la tabella sorgente

Aggiunta la sincronizzazione come task al job WhatIf

### 3. Creazione del FeatureSpec

In [0]:
# definiamo il lookup per estrarre tutte le features del record (univoco) identificato dalla look up key (ID)
feature_lookups = [
    FeatureLookup(
        table_name=f"{catalog}.whatif.output_palinsesto_passato_enriched",
        feature_names=None,  # Include all feature columns except lookup keys
        lookup_key=["Programma"],
    ),
]

# creiamo il feature spec con la look up dentro
fe.create_feature_spec(
    name=f"{catalog}.whatif.output_palinsesto_passato_enriched_features",
    features=feature_lookups
)

In [0]:
# testiamo che la registrazione del feature spec sia andato a buon fine
input_test_2 = pd.DataFrame(data = {'Programma' : ['LINEA VERDE']})
print(f"Test ID: {input_test_2}")

feature_set_test_2 = fe.create_training_set(
    df=spark.createDataFrame(input_test_2),                                 
    feature_spec=f"{catalog}.whatif.output_palinsesto_passato_enriched_features",
    label=None,
)

feature_set_test_2_df = feature_set_test_2.load_df()
print("Features:")
display(feature_set_test_2_df)

### 4. Creazione Feature Serving Endpoint
Su **Unity Catalog > catalog "{catalog}" > schema "what-id" > tab "Functions"**: selezionare la Feature Spec e poi cliccare in alto a destea su "Serve this Feature Spec"

### 5. Test Feature Serving Endpoint

In [0]:
client = mlflow.deployments.get_deploy_client("databricks")
response_test_2 = client.predict(endpoint="features_passato_enriched", inputs={"dataframe_records": input_test_2.to_dict(orient="records")})
response_test_2_df = pd.DataFrame(response_test_2["outputs"])
display(response_test_2_df)